In [4]:
from datetime import datetime
from pathlib import Path

import click
import numpy as np
import pandas as pd
import tensorflow as tf
from pydantic import PositiveInt
from rich.progress import Progress

# from tensorflow.keras import mixed_precision
from tqdm import tqdm

import migration.datasets_original_ds as datasets_original_ds
from migration.config import DatasetConfig, OptimizedBound, TrainingConfig
from migration.datasets import TensorflowEncodedBatchedDatasetBuilder
from migration.models import vrnn
from migration.models.vrnn_elbo import VRNN
from migration.models.vrnn_fivo import VRNNboundFIVO
from migration.utils import AverageMeter, console, logger


In [5]:
from migration.config import DatasetConfig, OptimizedBound, TrainingConfig
from migration.datasets import TensorflowEncodedBatchedDatasetBuilder
import numpy as np
def get_pandas_generator(parquet : Path):
    def get_in_memory_dataset_generator():
        _ds = pd.read_parquet(parquet).replace(1, 0.9999)
        _ds.insert(0, 'temp_id', range(0, len(_ds)))
        _ds = _ds.set_index('temp_id', append=True)
        _ds = _ds.sort_index().reset_index(1).drop('temp_id', axis=1)
        _idxs = _ds.index.unique()
        for idx in range(len(_idxs)):
            track = _ds.loc[_idxs[idx]].values
            yield track.reshape((-1, 4)).astype(np.float32)
    return get_in_memory_dataset_generator


In [9]:
def _get_config() -> TrainingConfig:
    return TrainingConfig(
    dataset=DatasetConfig(
        training_parquet='../../new_data/ais_train.parquet',
        validation_parquet='../../new_data/ais_val.parquet',
        test_parquet='../../new_data/ais_test.parquet',
        mean_pickle='../../data/ct_2017010203_10_20/mean.pkl',
        shuffle=False,
        val_size=15813 // 32,
        training_size=73795 // 32,
    ), epochs=9999)


In [10]:
cfg = _get_config().dataset

In [11]:
train_generator = get_pandas_generator(cfg.training_parquet)
train = TensorflowEncodedBatchedDatasetBuilder(
    track_generator=train_generator,
    batch_size=cfg.batch_size,
    lat_bins=cfg.encoding_bins.lat,
    lon_bins=cfg.encoding_bins.lon,
    sog_bins=cfg.encoding_bins.sog,
    cog_bins=cfg.encoding_bins.cog,
    shuffle=cfg.shuffle,
    repeat=True
).build()

I0000 00:00:1753188618.502615 1342869 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 41768 MB memory:  -> device: 0, name: NVIDIA L40S-48Q, pci bus id: 0000:00:05.0, compute capability: 8.9


In [12]:
it = iter(train)

In [17]:
_, targets, lengths = next(it)

In [18]:
targets.shape

TensorShape([132, 32, 702])

In [ ]:
tf.reduce_sum(targets, axis=1)

<tf.Tensor: shape=(132, 702), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

: 

In [42]:
lengths.shape

TensorShape([32])

In [43]:
max_seq_len = tf.reduce_max(input_tensor=lengths)


tf.transpose(
        a=tf.sequence_mask(lengths, maxlen=max_seq_len, dtype=tf.float32),
        perm=[1, 0])

<tf.Tensor: shape=(132, 32), dtype=float32, numpy=
array([[1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

In [20]:
d_idx_inbatch = 2

In [22]:
seq_len_d = lengths[d_idx_inbatch]
seq_len_d

<tf.Tensor: shape=(), dtype=int32, numpy=66>

In [28]:
targets[:seq_len_d,d_idx_inbatch,:].shape

TensorShape([66, 702])

In [29]:
targets[:seq_len_d,d_idx_inbatch,:]

<tf.Tensor: shape=(66, 702), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

In [33]:
len(np.nonzero(targets[:seq_len_d,d_idx_inbatch,:]))

2

In [41]:
tf.sequence_mask(tf.convert_to_tensor([7,2,4,5,6]), dtype=tf.float32)

<tf.Tensor: shape=(5, 7), dtype=float32, numpy=
array([[1., 1., 1., 1., 1., 1., 1.],
       [1., 1., 0., 0., 0., 0., 0.],
       [1., 1., 1., 1., 0., 0., 0.],
       [1., 1., 1., 1., 1., 0., 0.],
       [1., 1., 1., 1., 1., 1., 0.]], dtype=float32)>

In [39]:
tf.transpose(
            a=tf.sequence_mask(tf.convert_to_tensor([32,2,4,5,6]), dtype=tf.float32),
            perm=[1, 0])

<tf.Tensor: shape=(32, 5), dtype=float32, numpy=
array([[1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1.],
       [1., 0., 1., 1., 1.],
       [1., 0., 1., 1., 1.],
       [1., 0., 0., 1., 1.],
       [1., 0., 0., 0., 1.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.]], dtype=float32)>

In [36]:
np.nonzero(targets[:seq_len_d,d_idx_inbatch,:])

(array([ 0,  0,  0,  0,  1,  1,  1,  1,  2,  2,  2,  2,  3,  3,  3,  3,  4,
         4,  4,  4,  5,  5,  5,  5,  6,  6,  6,  6,  7,  7,  7,  7,  8,  8,
         8,  8,  9,  9,  9,  9, 10, 10, 10, 10, 11, 11, 11, 11, 12, 12, 12,
        12, 13, 13, 13, 13, 14, 14, 14, 14, 15, 15, 15, 15, 16, 16, 16, 16,
        17, 17, 17, 17, 18, 18, 18, 18, 19, 19, 19, 19, 20, 20, 20, 20, 21,
        21, 21, 21, 22, 22, 22, 22, 23, 23, 23, 23, 24, 24, 24, 24, 25, 25,
        25, 25, 26, 26, 26, 26, 27, 27, 27, 27, 28, 28, 28, 28, 29, 29, 29,
        29, 30, 30, 30, 30, 31, 31, 31, 31, 32, 32, 32, 32, 33, 33, 33, 33,
        34, 34, 34, 34, 35, 35, 35, 35, 36, 36, 36, 36, 37, 37, 37, 37, 38,
        38, 38, 38, 39, 39, 39, 39, 40, 40, 40, 40, 41, 41, 41, 41, 42, 42,
        42, 42, 43, 43, 43, 43, 44, 44, 44, 44, 45, 45, 45, 45, 46, 46, 46,
        46, 47, 47, 47, 47, 48, 48, 48, 48, 49, 49, 49, 49, 50, 50, 50, 50,
        51, 51, 51, 51, 52, 52, 52, 52, 53, 53, 53, 53, 54, 54, 54, 54, 55,
        55, 

In [35]:
np.nonzero(targets[:seq_len_d,d_idx_inbatch,:])[1]

array([189, 373, 606, 658, 189, 375, 606, 658, 189, 376, 606, 658, 189,
       377, 606, 658, 190, 379, 606, 658, 190, 380, 606, 659, 190, 381,
       606, 658, 190, 383, 607, 659, 190, 384, 606, 659, 191, 385, 606,
       659, 191, 387, 606, 659, 191, 388, 606, 659, 191, 389, 607, 659,
       192, 391, 605, 658, 193, 392, 604, 658, 194, 393, 604, 658, 195,
       394, 604, 658, 196, 395, 605, 659, 197, 396, 602, 656, 198, 397,
       603, 658, 199, 398, 603, 658, 201, 398, 603, 658, 202, 399, 603,
       658, 203, 400, 603, 658, 204, 401, 604, 659, 205, 403, 605, 659,
       206, 404, 605, 659, 207, 405, 604, 658, 208, 406, 604, 658, 209,
       407, 603, 657, 210, 408, 603, 658, 212, 409, 603, 658, 213, 409,
       603, 658, 214, 410, 603, 658, 216, 411, 603, 658, 217, 412, 604,
       657, 218, 413, 604, 658, 219, 414, 604, 658, 220, 415, 604, 657,
       221, 416, 603, 657, 222, 417, 603, 657, 223, 418, 604, 657, 224,
       419, 604, 657, 225, 420, 604, 657, 226, 421, 604, 656, 22

In [26]:
np.nonzero(targets[:seq_len_d,d_idx_inbatch,:])[1].reshape(-1,4)

array([[189, 373, 606, 658],
       [189, 375, 606, 658],
       [189, 376, 606, 658],
       [189, 377, 606, 658],
       [190, 379, 606, 658],
       [190, 380, 606, 659],
       [190, 381, 606, 658],
       [190, 383, 607, 659],
       [190, 384, 606, 659],
       [191, 385, 606, 659],
       [191, 387, 606, 659],
       [191, 388, 606, 659],
       [191, 389, 607, 659],
       [192, 391, 605, 658],
       [193, 392, 604, 658],
       [194, 393, 604, 658],
       [195, 394, 604, 658],
       [196, 395, 605, 659],
       [197, 396, 602, 656],
       [198, 397, 603, 658],
       [199, 398, 603, 658],
       [201, 398, 603, 658],
       [202, 399, 603, 658],
       [203, 400, 603, 658],
       [204, 401, 604, 659],
       [205, 403, 605, 659],
       [206, 404, 605, 659],
       [207, 405, 604, 658],
       [208, 406, 604, 658],
       [209, 407, 603, 657],
       [210, 408, 603, 658],
       [212, 409, 603, 658],
       [213, 409, 603, 658],
       [214, 410, 603, 658],
       [216, 4